# Precificação de Imóveis no Brasil: Análise Exploratória e Modelagem Preditiva

Este projeto utiliza uma **base de dados real de 10.692 anúncios de aluguel de imóveis** no Brasil, cobrindo as cidades de **São Paulo, Rio de Janeiro, Belo Horizonte, Porto Alegre e Campinas**.

### Objetivos do Estudo:
1. **Análise Exploratória (EDA):** examinar os valores reais de aluguel, custo por metro quadrado e diferenças de mercado entre as capitais.
2. **Limpeza e Tratamento de Outliers:** identificar inconsistências e valores extremos comuns em dados reais de mercado.
3. **Engenharia de Variáveis e Prevenção de Data Leakage:** isolar apenas os atributos estruturais e de localização conhecidos previamente à locação.
4. **Treinamento e Comparação de Modelos de Machine Learning:** comparar uma regressão linear regularizada (Ridge) com Random Forest Regressor.
5. **Interpretabilidade e Simulação:** analisar os fatores que mais valorizam o aluguel e estimar o preço de novos anúncios.

### 1. Importação de Bibliotecas

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

### 2. Leitura e Visão Geral da Base Real

In [ ]:
caminho_csv = os.path.join('dados', 'imoveis_brasil.csv')
df = pd.read_csv(caminho_csv)

print(f'Total de imóveis na base bruta: {df.shape[0]:,} anúncios')
print(f'Colunas disponíveis: {list(df.columns)}')
df.head()

In [ ]:
# Distribuição de anúncios por cidade
print('Anúncios por cidade:')
print(df['cidade'].value_counts())

print('\nEstatísticas descritivas (dados brutos):')
df[['area_m2', 'quartos', 'banheiros', 'vagas_garagem', 'valor_condominio', 'valor_aluguel', 'valor_iptu']].describe().round(1)

### 3. Tratamento de Outliers Extremos
Em bases reais de imóveis, anúncios com erros de digitação (ex: área de 40.000 m² ou condomínio de R$ 1.000.000) distorcem as métricas dos modelos lineares. Vamos aplicar um filtro conservador para remover anomalias extremas mantendo 99.5% dos dados válidos.

In [ ]:
limite_area = 800
limite_condo = 25000
limite_aluguel = 25000
limite_iptu = 15000

filtro = (
    (df['area_m2'] <= limite_area) &
    (df['valor_condominio'] <= limite_condo) &
    (df['valor_aluguel'] <= limite_aluguel) &
    (df['valor_iptu'] <= limite_iptu)
)

df_limpo = df[filtro].copy()
print(f'Imóveis antes do filtro: {len(df):,}')
print(f'Imóveis após filtro:     {len(df_limpo):,} ({len(df_limpo)/len(df)*100:.1f}% retidos)')

### 4. Análise Exploratória (EDA)

In [ ]:
# Preço médio do aluguel por m² por cidade
df_limpo['preco_m2'] = df_limpo['valor_aluguel'] / df_limpo['area_m2']
media_m2 = df_limpo.groupby('cidade')['preco_m2'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=media_m2.values, y=media_m2.index, palette='Blues_r', ax=ax)
ax.set_title('Preço Médio Real do Aluguel por m² (R$ / m²)', fontsize=14, pad=12)
ax.set_xlabel('R$ / m²')
ax.set_ylabel('')
for i, val in enumerate(media_m2.values):
    ax.text(val + 0.5, i, f'R$ {val:.2f}', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Relação entre Área Útil e Valor do Aluguel
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df_limpo,
    x='area_m2',
    y='valor_aluguel',
    hue='cidade',
    alpha=0.5,
    palette='tab10',
    ax=ax
)
ax.set_title('Área Útil vs. Valor do Aluguel por Cidade', fontsize=14, pad=12)
ax.set_xlabel('Área Útil (m²)')
ax.set_ylabel('Aluguel Mensal (R$)')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de Correlação entre Atributos Numéricos
cols_corr = ['area_m2', 'quartos', 'banheiros', 'vagas_garagem', 'andar', 'valor_condominio', 'valor_iptu', 'valor_aluguel']
matriz_corr = df_limpo[cols_corr].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt='.2f', square=True, cbar=True)
plt.title('Matriz de Correlação de Pearson', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

### 5. Preparação para Machine Learning (Sem Vazamento de Dados)

> **Nota Importante sobre Data Leakage:** as colunas `valor_seguro_incendio` e `valor_total` são derivadas matematicamente do aluguel já definido. Em um caso real de previsão para um novo imóvel anunciado, essas taxas não são conhecidas previamente. Portanto, excluímos ambas das variáveis preditoras para garantir um modelo realista.

In [ ]:
target = 'valor_aluguel'
features_num = ['area_m2', 'quartos', 'banheiros', 'vagas_garagem', 'andar', 'valor_condominio', 'valor_iptu']
features_cat = ['cidade', 'aceita_animais', 'mobiliado']

# One-Hot Encoding nas variáveis categóricas
df_ml = pd.get_dummies(df_limpo[features_num + features_cat + [target]], columns=features_cat, drop_first=True)

X = df_ml.drop(columns=[target])
y = df_ml[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f'Amostras de Treino: {len(X_train):,}')
print(f'Amostras de Teste:  {len(X_test):,}')
print(f'Features preditoras: {X_train.shape[1]}')

### 6. Treinamento e Comparativo de Modelos

In [ ]:
# Modelo 1: Regressão Linear com Regularização L2 (Ridge)
modelo_linear = Ridge(alpha=1.0)
modelo_linear.fit(X_train, y_train)
y_pred_linear = modelo_linear.predict(X_test)

# Modelo 2: Random Forest Regressor (Árvores de Decisão Encadeadas)
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)

# Função de cálculo das métricas
def avaliar_modelo(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    return mae, rmse, r2

mae_lin, rmse_lin, r2_lin = avaliar_modelo(y_test, y_pred_linear)
mae_rf, rmse_rf, r2_rf = avaliar_modelo(y_test, y_pred_rf)

tabela_comparativa = pd.DataFrame({
    'Modelo': ['Regressão Ridge (Linear)', 'Random Forest Regressor'],
    'MAE (Erro Médio Absoluto)': [f'R$ {mae_lin:,.2f}', f'R$ {mae_rf:,.2f}'],
    'RMSE (Raiz do Erro Médio)': [f'R$ {rmse_lin:,.2f}', f'R$ {rmse_rf:,.2f}'],
    'R² (Poder Explicativo)': [f'{r2_lin:.3f}', f'{r2_rf:.3f}']
})

tabela_comparativa

### 7. Diagnóstico e Importância dos Atributos

In [ ]:
# Gráfico Real vs. Previsto no conjunto de teste
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred_rf, alpha=0.3, color='#1f77b4', edgecolors='none')
limite_grafico = [0, max(y_test.max(), y_pred_rf.max()) * 1.05]
ax.plot(limite_grafico, limite_grafico, color='red', linestyle='--', linewidth=2, label='Linha de Perfeição (y = x)')
ax.set_xlim(limite_grafico)
ax.set_ylim(limite_grafico)
ax.set_xlabel('Valor Real do Aluguel (R$)')
ax.set_ylabel('Valor Previsto pelo Modelo (R$)')
ax.set_title('Random Forest: Aluguel Real vs. Aluguel Previsto', fontsize=14, pad=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Importância das variáveis preditoras no Random Forest
importancias = pd.Series(modelo_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=importancias.values, y=importancias.index, palette='viridis', ax=ax)
ax.set_title('Importância Relativa de Cada Atributo na Previsão do Aluguel', fontsize=14, pad=12)
ax.set_xlabel('Importância (Critério de Gini)')
plt.tight_layout()
plt.show()

### 8. Simulação Prática de Previsão de Aluguel
Podemos utilizar o modelo treinado para estimar o valor de mercado de um imóvel que ainda não foi anunciado.

In [ ]:
# Cenário: Apartamento de 75m² em São Paulo, mobiliado, 2 quartos, 2 banheiros, 1 vaga, andar 6
simulacao = pd.DataFrame(columns=X_train.columns, index=[0]).fillna(0)

simulacao['area_m2'] = 75
simulacao['quartos'] = 2
simulacao['banheiros'] = 2
simulacao['vagas_garagem'] = 1
simulacao['andar'] = 6
simulacao['valor_condominio'] = 650
simulacao['valor_iptu'] = 120
if 'cidade_São Paulo' in simulacao.columns:
    simulacao['cidade_São Paulo'] = 1
if 'mobiliado_Sim' in simulacao.columns:
    simulacao['mobiliado_Sim'] = 1
if 'aceita_animais_Sim' in simulacao.columns:
    simulacao['aceita_animais_Sim'] = 1

preco_estimado = modelo_rf.predict(simulacao)[0]
print(f'Aluguel sugerido para o imóvel simulado: R$ {preco_estimado:,.2f}')